# qBraid Challenge Demo: Compiler-Aware Quantum Benchmarking

This notebook presents a **portable QAOA MaxCut benchmark** built with qBraid.

We compare two qBraid compilation strategies and evaluate the quality-vs-cost tradeoff across two execution environments:
1. Ideal simulator
2. Noisy constrained simulator

## 1) Imports and Benchmark Entry Point
This notebook reuses the tested implementation from `benchmark_qbraid_qaoa.py`.

In [1]:
from pathlib import Path
import json

from benchmark_qbraid_qaoa import benchmark

## 2) Run the Benchmark
You can increase shots/grid-points for final reporting.

In [2]:
report = benchmark(shots=2048, grid_points=6)
report

c:\Users\nicow\OneDrive\Documents\Job\projects\QuantumHackathon\qml_env\Lib\site-packages\qiskit\compiler\transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


{'algorithm': 'QAOA MaxCut (p=1)',
 'source_framework': 'qiskit',
 'graph_nodes': 4,
 'graph_edges': 5,
 'maxcut_optimum': 4.0,
 'selected_parameters': {'gamma': 1.2766370614359173,
  'beta': 0.6483185307179585},
 'strategy_results': [{'name': 'A_qasm2_constrained',
   'qbraid_target': 'qasm2',
   'depth': 18,
   'two_qubit_count': 10,
   'width': 4,
   'ideal_success_prob': 0.595703125,
   'ideal_approx_ratio': 0.80755615234375,
   'noisy_success_prob': 0.41162109375,
   'noisy_approx_ratio': 0.7213134765625},
  {'name': 'B_qasm3_flexible',
   'qbraid_target': 'qasm3',
   'depth': 18,
   'two_qubit_count': 10,
   'width': 4,
   'ideal_success_prob': 0.56103515625,
   'ideal_approx_ratio': 0.79150390625,
   'noisy_success_prob': 0.41552734375,
   'noisy_approx_ratio': 0.7166748046875}],
 'best_tradeoff_strategy': 'A_qasm2_constrained',
 'tradeoff_definition': 'noisy_approx_ratio / max(two_qubit_count,1)'}

## 3) Save Results Artifact
This cell writes the same JSON artifact used by the script workflow.

In [3]:
out_path = Path('results/benchmark_results.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
out_path

WindowsPath('results/benchmark_results.json')

## 4) Presentation Table
A compact side-by-side summary for slides.

In [4]:
rows = report['strategy_results']
header = (
    f"{'Strategy':<22} {'Target':<8} {'Depth':>6} {'2Q':>4} {'Ideal Ratio':>12} {'Noisy Ratio':>12} {'Ideal Succ':>11} {'Noisy Succ':>11}"
)
print(header)
print('-' * len(header))
for r in rows:
    print(
        f"{r['name']:<22} {r['qbraid_target']:<8} {r['depth']:>6} {r['two_qubit_count']:>4} {r['ideal_approx_ratio']:>12.4f} {r['noisy_approx_ratio']:>12.4f} {r['ideal_success_prob']:>11.4f} {r['noisy_success_prob']:>11.4f}"
    )

print('\nBest tradeoff strategy:', report['best_tradeoff_strategy'])
print('Tradeoff definition:', report['tradeoff_definition'])

Strategy               Target    Depth   2Q  Ideal Ratio  Noisy Ratio  Ideal Succ  Noisy Succ
---------------------------------------------------------------------------------------------
A_qasm2_constrained    qasm2        18   10       0.8076       0.7213      0.5957      0.4116
B_qasm3_flexible       qasm3        18   10       0.7915       0.7167      0.5610      0.4155

Best tradeoff strategy: A_qasm2_constrained
Tradeoff definition: noisy_approx_ratio / max(two_qubit_count,1)


## 5) Required Challenge Answers (Notebook Form)
- **What algorithm did you implement?** QAOA MaxCut (p=1) with coarse parameter tuning.
- **What was your source representation?** Qiskit `QuantumCircuit`.
- **How did qBraid transform the workload?** qBraid transpiled source circuits to alternate targets and back for comparable execution.
- **What two compilation strategies did you compare?** Constrained `qasm2` path vs flexible `qasm3` path with different conversion limits.
- **What changed in the compiled programs?** Resource profile (depth, 2-qubit gates) and noisy-environment quality.
- **Which strategy best preserved performance?** See `best_tradeoff_strategy` above.
- **What was the cost of that strategy?** See depth/2Q/width in `strategy_results`.